In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive
 20191216_122821.mp4		  logic
 20200101_074036.mp4		  math
 20200324_211649.mp4		  open_problem
 Classroom			 'Screenshot_20220307-174728_().jpg'
'Colab Notebooks'		  대마도여행250224.gmap
'Gantt Chart_07의 사본.gslides'   여수여행_241216.gmap
 gptneo-1.3B-finetuned		 '역사 보고서.show'
 gptneo-1.3B-masked		  오키나와여행_241230.gmap
 gptneo-finetuned-qa		  통지서.html
 knowledge			  홍콩여행_240121.gmap


In [ ]:
# 1. 필요한 패키지 설치
!pip install --upgrade transformers datasets accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.4/491.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.


In [ ]:
import json
import torch
from datasets import Dataset
from transformers import GPT2LMHeadModel, PreTrainedTokenizerFast, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [ ]:
# 모델 및 토크나이저 로딩
model_name = "skt/kogpt2-base-v2"
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.add_special_tokens({'eos_token': '</s>', 'pad_token': '<pad>'})
model.resize_token_embeddings(len(tokenizer))

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


Embedding(51200, 768)

In [ ]:
# 질문-답변 데이터 로드
from google.colab import files
import json

uploaded = files.upload()

with open("FAQ.json", "r", encoding="utf-8") as f:  # FAQ->augmented_qa
    raw_data = json.load(f)

Saving FAQ.json to FAQ.json


In [ ]:
# KoGPT 학습용 포맷 구성 (나중에 이거 지우고)
formatted_data = [{"text": f"답변: {item['Answer']}\n"} for item in raw_data]
dataset = Dataset.from_list(formatted_data)

In [ ]:
# KoGPT 학습용 포맷 구성 (이걸로 교체)
formatted_data = [{"text": f"질문: {item['Question']}\n답변: {item['Answer']}"} for item in raw_data]
dataset = Dataset.from_list(formatted_data)

In [ ]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)

In [ ]:
tokenized_dataset = dataset.map(tokenize, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Map:   0%|          | 0/21 [00:00<?, ? examples/s]

In [ ]:
# 학습 인자 설정
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/KoGPT/CheckPoint/Q&A/Response",
    overwrite_output_dir=True,
    num_train_epochs=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    logging_steps=25,
    save_steps=250,
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    learning_rate=5e-5,
    warmup_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

<ipython-input-21-f686d387b397>:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 학습 시작
trainer.train()

Step,Training Loss
25,1.283500
50,0.442700
75,0.180000
100,0.136200
125,0.096100
150,0.084700
175,0.076000
200,0.066400
225,0.068700
250,0.063700


TrainOutput(global_step=500, training_loss=0.15613326597213745, metrics={'train_runtime': 171.5984, 'train_samples_per_second': 12.238, 'train_steps_per_second': 2.914, 'total_flos': 457522348032000.0, 'train_loss': 0.15613326597213745, 'epoch': 83.38095238095238})

In [ ]:
# 모델 저장
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Q&A/Response")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/KoGPT/Model/Q&A/Response")

('/content/drive/MyDrive/KoGPT/tokenizer_config.json',
 '/content/drive/MyDrive/KoGPT/special_tokens_map.json',
 '/content/drive/MyDrive/KoGPT/tokenizer.json')